# Análise das compras com Apriori

**Projeto:** rede de farmácias — Banco de Dados II.

**Objetivo:** identificar pares de produtos frequentes e gerar regras de associação que possam apoiar decisões de negócio.

A análise deve ser executada após a carga pelo ETL. As conclusões serão preenchidas com as evidências obtidas nos dados do projeto.

## Preparação e parâmetros

Cada venda é uma cesta. Os produtos são identificados pelos códigos; repetições do mesmo produto em uma compra são removidas. A matriz é booleana e inclui compras com apenas um produto.

Os valores iniciais são suporte de **2%**, confiança de **60%**, lift **maior que 1** e conjuntos com no máximo **dois produtos**. A justificativa final deve considerar a distribuição dos dados e os resultados das execuções.

Suporte = proporção de compras com A e B; confiança = proporção de compras com B entre as compras com A; lift = confiança dividida pela frequência de B. As regras mostram associações observadas.

In [9]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

candidatas = [Path.cwd(), Path.cwd() / 'mineracao']
PASTA = next((p for p in candidatas if (p / 'executar_apriori.py').is_file()), None)
if PASTA is None:
    raise RuntimeError('Abra o notebook na pasta mineracao ou na raiz do projeto.')

BASE = PASTA / 'dados_preparados' / 'compras.csv'
RESULTADOS = PASTA / 'resultados_apriori'

SUPORTE = 0.02
CONFIANCA = 0.60
LIFT = 1.0
EXECUCAO_CONCLUIDA = False

print('Python:', sys.version.split()[0])
print('Parâmetros:', {'suporte': SUPORTE, 'confianca': CONFIANCA, 'lift_maior_que': LIFT, 'tamanho_maximo': 2})

Python: 3.13.17
Parâmetros: {'suporte': 0.02, 'confianca': 0.6, 'lift_maior_que': 1.0, 'tamanho_maximo': 2}


## Base utilizada

Gere os dados pelo script `preparar_dados.py`. Confira as contagens e o período da carga com a equipe de ETL. Confirme também o tratamento de vendas canceladas.

As linhas abaixo representam produtos presentes nas compras. O identificador da venda agrupa os itens em cestas.

In [10]:
BASE_DISPONIVEL = False
if BASE.is_file():
    dados = pd.read_csv(
        BASE,
        dtype={'codigo_venda': 'string', 'codigo_produto': 'string', 'produto': 'string'},
        encoding='utf-8-sig',
        keep_default_na=False,
    )
    BASE_DISPONIVEL = not dados.empty
    print('Linhas na base:', len(dados))
    print('Compras distintas:', dados['codigo_venda'].nunique())
    print('Produtos distintos:', dados['codigo_produto'].nunique())
    display(dados.head(10))
else:
    print('Base ainda não gerada. Após o ETL, execute preparar_dados.py e rode novamente o notebook.')

Base ainda não gerada. Após o ETL, execute preparar_dados.py e rode novamente o notebook.


## Execução do algoritmo

A célula executa `executar_apriori.py` com os parâmetros acima. O script reconstrói as cestas, calcula as combinações frequentes e seleciona as regras. Os arquivos de saída preservam a base e o resumo da execução.

In [11]:
EXECUCAO_CONCLUIDA = False
if BASE_DISPONIVEL:
    comando = [
        sys.executable, str(PASTA / 'executar_apriori.py'),
        '--base', str(BASE), '--saida', str(RESULTADOS),
        '--suporte', str(SUPORTE), '--confianca', str(CONFIANCA), '--lift', str(LIFT),
    ]
    ambiente = dict(os.environ, PYTHONIOENCODING='utf-8')
    execucao = subprocess.run(
        comando, capture_output=True, text=True, encoding='utf-8', env=ambiente,
    )
    print(execucao.stdout)
    if execucao.returncode:
        print(execucao.stderr)
        execucao.check_returncode()
    EXECUCAO_CONCLUIDA = True
else:
    print('Execução pendente: a base de compras ainda não está disponível.')

Execução pendente: a base de compras ainda não está disponível.


## Resultados e rastreabilidade

A tabela mostra as regras que atenderam aos critérios. Os indicadores `suporte` e `confianca` usam escala de 0 a 1. O resumo registra quantidades, parâmetros, versões e o SHA-256 da base.

Quando nenhuma regra é selecionada, registre a ausência e analise as características da base e os limites usados.

In [12]:
if EXECUCAO_CONCLUIDA:
    resumo = json.loads((RESULTADOS / 'parametros_e_resumo.json').read_text(encoding='utf-8'))
    regras = pd.read_csv(RESULTADOS / 'regras_de_associacao.csv', encoding='utf-8-sig')
    combinacoes = pd.read_csv(RESULTADOS / 'combinacoes_frequentes.csv', encoding='utf-8-sig')
    print(json.dumps(resumo, ensure_ascii=False, indent=2))
    print('\nCombinações frequentes (até 20 linhas):')
    display(combinacoes.head(20))
    print('\nRegras selecionadas:')
    display(regras)
    if regras.empty:
        print('Nenhuma regra atendeu aos critérios desta execução.')
else:
    print('Resultados pendentes da execução com a base do projeto.')

Resultados pendentes da execução com a base do projeto.


## Conclusões de negócio — preencher após a execução

**Contexto da base:** informar período analisado, quantidade de compras e produtos, origem dos dados e tratamento de cancelamentos.

### 1. Primeira conclusão
- **Achado observado:** preencher.
- **Evidência:** identificar produtos, contagens e indicadores aplicáveis.
- **Ação de negócio sugerida:** preencher.
- **Limitação:** preencher.

### 2. Segunda conclusão
- **Achado observado:** preencher.
- **Evidência:** identificar produtos, contagens e indicadores aplicáveis.
- **Ação de negócio sugerida:** preencher.
- **Limitação:** preencher.

### 3. Terceira conclusão
- **Achado observado:** preencher.
- **Evidência:** identificar produtos, contagens e indicadores aplicáveis.
- **Ação de negócio sugerida:** preencher.
- **Limitação:** preencher.

As conclusões devem se apoiar nos resultados obtidos, inclusive quando não houver regras consistentes. As três conclusões não precisam corresponder a três regras diferentes.

## Referências técnicas

- [Apriori — mlxtend](https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/)
- [Regras de associação — mlxtend](https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/)
